# Sequential offers: a weekly retail campaign

A weekly offer changes who stays engaged, and engagement steers the next offer. This notebook
estimates the full-price order rate of five offer policies with longitudinal TMLE. It then fits the
same data with three other estimators and compares them. Each step shows its code, its output, and
what the output tells you. [Longitudinal TMLE](../technical-reference/longitudinal-tmle.md) gives
the sequential regression and the algorithm. [Sequential g-computation and
IPW](../technical-reference/sequential-baselines.md) gives the two baselines.

## The applied question

A subscription retailer sends one offer to each customer in each of three weeks. The offer is no
code, a 15% discount code, or a free-shipping code. Before each send, the CRM records whether the
customer is engaged that week. The CRM draws each offer at random, with logged probabilities that
depend on that engagement. Customers who withdraw tracking consent are lost to follow-up.

The retention lead asks one question. What share of customers would place a full-price order in
week 4 under each declared offer policy, and which policy beats which? A full-price order is an
order placed without a code. Every policy is a plan of three offers, not the effect of one send.

This page uses a synthetic law. [The examples index](index.md#the-retailer-program) states how it
relates to the health program of the other tutorials.

## What you will learn

| after this notebook you can | the step that shows it |
| --- | --- |
| read a long customer panel, with the logged offer probabilities as the known mechanism | Steps 2 and 5 |
| recognize engagement as a confounder that an earlier offer changed | Step 3 |
| declare three offer levels and a rule that reads the weekly engagement | Step 5 |
| fit longitudinal TMLE with explicit learners, and compare four estimators | Steps 6 to 8 |
| compare two policies with the joint influence curve | Step 9 |
| read stagewise support, a truncation curve, and a synthetic deployment check | Steps 10 to 12 |
| aim a covariate benchmark at the assumption that is at risk | Step 13 |

## Why this method

Engagement is a time-varying confounder affected by prior treatment. It also drives consent loss.

| engagement in this law | what a single regression does with it |
| --- | --- |
| a discount raises it, and it raises the full-price probability | adjusting for it blocks the part of the discount effect that runs through engagement |
| it steers the next offer | leaving it out leaves the next offer confounded |
| it raises the chance that the customer stays consenting | a complete-case analysis keeps a selected group |

Sequential regression works backward from the week-4 outcome. Each week's regression conditions on
the history available at that week. The recursion then averages that history under the policy.
This evaluates the g-formula. The table defines the terms this page uses most.

| term | plain meaning |
| --- | --- |
| node | one decision time. Here, the weekly send in weeks 1, 2, and 3 |
| history | every column recorded before the send at a node |
| plan | a rule for each node. An entry is one offer for everyone, or a rule that reads the history |
| win-back | the plan that sends the discount to a customer who is not engaged, and no offer otherwise |
| declared offer mechanism | the logged probability of each offer, read from columns. The fit runs no offer model |
| nuisance | a model the estimate needs but the question does not ask about. Here, the outcome, pseudo-outcome, and consent models |
| targeting | a small update to each node's regression, weighted by one over the cumulative probability of following the plan and staying consenting, that removes first-order bias |
| influence curve | how much each customer moves the estimate. Its variance gives the standard error |
| sequential positivity | at every node, each history that the plan can reach has some chance of the plan's offer and of staying consenting |

## Step 1: set up

The setup imports the libraries and prints the installed `cleverly` version. Every fit below passes
its learners, fold count, and random seed explicitly, so a rerun reproduces the stored outputs.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression

import cleverly

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 20)
print("cleverly", cleverly.__version__)

cleverly 0.1.3


**What this output tells you.** The stored outputs in this notebook came from the version named
above. A different version can print different numbers.

## Step 2: the data

The generator `make_sequential_offers` returns a long panel with one row per customer and week. A
customer who withdraws consent has no later rows. The code prints the shape and the column names.
It counts the customers who withdrew consent, and it prints the true values of the synthetic law.

In [2]:
from cleverly.datasets import make_sequential_offers

panel, truth = make_sequential_offers(20_000, seed=3)
print("rows and columns:", panel.shape)
print("customers:", panel["customer"].nunique())
print(list(panel.columns))
print(panel.head(6).round(3))
print()
left = (panel.groupby("customer")["consent"].min() == 0).sum()
print(f"customers who withdrew consent before week 4: {left} of {panel['customer'].nunique()}")
print("known values of the synthetic law:")
for name, value in truth.items():
    print(f"  {name:36s} {value:.4f}")

rows and columns: (57079, 10)
customers: 20000
['customer', 'week', 'tenure', 'engaged', 'offer', 'consent', 'p_none', 'p_discount', 'p_shipping', 'full_price']
   customer  week  tenure  engaged     offer  consent  p_none  p_discount  p_shipping  full_price
0         0     1   2.041        0  discount        1     0.3        0.45        0.25         1.0
1         0     2   2.041        1  shipping        1     0.4        0.25        0.35         1.0
2         0     3   2.041        1      none        1     0.4        0.25        0.35         1.0
3         1     1  -2.556        0  discount        1     0.3        0.45        0.25         0.0
4         1     2  -2.556        1      none        1     0.4        0.25        0.35         0.0
5         1     3  -2.556        0  discount        1     0.3        0.45        0.25         0.0

customers who withdrew consent before week 4: 2697 of 20000
known values of the synthetic law:
  ey_regimen[none]                     0.5275
  ey_regime

**What this output tells you.** The panel has 57079 rows and 10 columns for 20000 customers. Each
customer has up to three rows.

| column | role |
| --- | --- |
| `customer`, `week` | the unit and the decision time |
| `tenure` | a baseline covariate, a standard-normal draw |
| `engaged` | the engagement score of that week, recorded before the offer |
| `offer` | the offer sent that week: `none`, `discount`, or `shipping` |
| `consent` | 1 if the customer still allows tracking after that send, and 0 if the customer withdrew |
| `p_none`, `p_discount`, `p_shipping` | the logged probabilities of the three offers, for that customer and week |
| `full_price` | the week-4 outcome. It repeats on each row, and it is missing for a customer who withdrew |

The CRM chose the probabilities, so they are known. Engaged customers have 0.40 for no offer and
0.25 for a discount. Customers who are not engaged have 0.30 and 0.45.

A total of 2697 customers withdrew consent before week 4, and they have no outcome. The estimator
expects that shape. A complete-case frame would discard those customers before any model sees them.

The true values come from `offers_truth()`, which enumerates the law and does not simulate it.
The true full-price rate is 0.5275 with no offers, 0.4596 with a discount every week, and 0.5865
with free shipping every week. Win-back gives 0.5997, and win-back then ship gives 0.6268.

Free shipping beats no offers, and a discount every week loses to no offers. Win-back beats both
static policies. The true contrast of win-back against no offers is 0.0722. A real program has no
`truth`. Every comparison against it below is a teaching device.

## Step 3: association first

A confounder is a variable that changes both the next offer and the outcome. Here the confounder of
weeks 2 and 3 also responds to the earlier offer. The code reads three relations in the logs. It
then compares the customers who happened to follow each plan in all three weeks with the customers
who followed no offers. The comparison keeps only the customers who still consented after week 3.

In [3]:
from cleverly.datasets import OFFER_PLANS

week_one = panel[panel["week"] == 1]
week_two = panel[panel["week"] == 2][["customer", "engaged"]]
moved = week_one.merge(week_two, on="customer", suffixes=("", "_week2"))
print("share engaged in week 2, by the week-1 offer:")
print(moved.groupby("offer")["engaged_week2"].mean().round(3).to_string())
print()
print("share of each week-1 offer, by week-1 engagement:")
print(pd.crosstab(week_one["engaged"], week_one["offer"], normalize="index").round(3).to_string())
print()
print("share still consenting after the week-1 send, by week-1 engagement:")
print(week_one.groupby("engaged")["consent"].mean().round(3).to_string())
print()

wide = panel.pivot(index="customer", columns="week")
kept = wide[wide[("consent", 3)] == 1]
rows = []
for name, rule in OFFER_PLANS.items():
    followed = np.ones(len(kept), dtype=bool)
    for week in (1, 2, 3):
        engaged = kept[("engaged", week)].astype(int).to_numpy()
        followed &= kept[("offer", week)].to_numpy() == np.asarray(rule(engaged), dtype=object)
    rows.append(
        {
            "plan": name,
            "followers": int(followed.sum()),
            "share": kept[("full_price", 1)][followed].mean(),
        }
    )
crude = pd.DataFrame(rows).set_index("plan")
crude["crude_vs_none"] = crude["share"] - crude.loc["none", "share"]
crude["true_vs_none"] = [truth.get(f"ate_regimen[{name} vs none]", 0.0) for name in crude.index]
print(f"customers still consenting after week 3: {len(kept)} of {len(wide)}")
print(crude.round(3).to_string())

share engaged in week 2, by the week-1 offer:
offer
discount    0.889
none        0.550
shipping    0.668

share of each week-1 offer, by week-1 engagement:
offer    discount   none  shipping
engaged                           
0           0.455  0.304     0.241
1           0.248  0.406     0.345

share still consenting after the week-1 send, by week-1 engagement:
engaged
0    0.928
1    0.967

customers still consenting after week 3: 17303 of 20000
               followers  share  crude_vs_none  true_vs_none
plan                                                        
none                 718  0.578          0.000         0.000
discount             427  0.494         -0.084        -0.068
shipping             525  0.613          0.035         0.059
win_back            1270  0.617          0.039         0.072
win_back_ship        928  0.624          0.046         0.099


**What this output tells you.** The first table shows that the offer moves engagement. After a
discount, a share of 0.889 of the customers are engaged in week 2. After no offer the share is
0.550, and after free shipping it is 0.668.

The second table shows that engagement steers the offer. Engaged customers receive no offer at a
share of 0.406 and a discount at 0.248. Customers who are not engaged receive a discount at 0.455.
The third table shows that engagement predicts consent. The share still consenting is 0.967 for
engaged customers and 0.928 for the others.

Engagement is therefore a confounder of the later offers, and the earlier offer changes it. One
outcome regression cannot condition on it without blocking the path from the offer through
engagement. It cannot leave it out without confounding the later offer.

The last table is the crude comparison. It keeps 17303 of 20000 customers, those who consented
through week 3. Only 718 of them received no offer in all three weeks, and 427 received a discount
three times. The crude contrast of win-back against no offers is 0.039, and the true contrast is
0.072. For a discount every week the crude contrast is -0.084 and the true contrast is -0.068.

The crude comparison uses small follower groups. The offer probabilities depend on engagement, so
the followers of a plan are a covariate-dependent group. The comparison also drops customers who
withdrew. This is one draw, and the sizes of the misses are not general.

## Step 4: write the protocol

A `StudyProtocol` records the scientific design before any model runs.
[Point-treatment TMLE](point-treatment-tmle.ipynb#step-4-write-the-protocol) introduces its fields
and its fingerprint. The helper `offers_protocol()` returns the record for this campaign. It names the five
policies of the question, and it states how the design treats consent withdrawal and expired codes.

In [4]:
from cleverly.datasets import offers_protocol

protocol = offers_protocol()
print("\n".join(protocol.summary_lines()))

causal study protocol: schema 1; 94024dfd011817af
target population: Active subscribers at the start of the campaign window
eligibility: ['Active subscription at the first weekly send', 'Marketing consent at the first weekly send', 'At least one prior order', 'One account per customer, so shared-household accounts are excluded', 'An account that stays open through week 4, so a closure-free window is a condition of the target population']
time zero: The first weekly send, after the week-1 engagement score and before the first offer
treatment strategies: ['Send no offer in weeks 1 to 3', 'Send the 15% discount code every week', 'Send the free-shipping code every week', 'Win-back: send the discount code when the customer is not engaged, otherwise send no offer', 'Win-back then ship: send the discount code when the customer is not engaged, otherwise send the free-shipping code']
treatment versions: ['No code sent', 'The declared 15% code, one send per week, valid until the end of that week

**What this output tells you.** The first line gives the schema version and the fingerprint
`94024dfd011817af`. The other lines repeat each field. Read them as a checklist.

| protocol field | the question it answers for this campaign |
| --- | --- |
| target population and eligibility | who the effect is about. Eligibility includes an account that stays open through week 4, so a closure-free window is a condition of the population |
| time zero | the first weekly send, after the week-1 engagement score and before the first offer |
| treatment strategies and versions | the five policies, each with the declared code template. Each code is valid until the end of its week |
| outcome and horizon | a full-price order in week 4 |
| intercurrent-event handling | consent withdrawal censors the customer under a hypothetical strategy: the order had tracking continued. A code expires at the end of its week, so no code is redeemable in week 4 |
| interference unit | the individual customer |
| assumption rationale | randomization with logged probabilities gives sequential exchangeability for the offers, given tenure and the weekly engagement. Tenure and engagement are the recorded causes of consent loss |

The last row names the confounders. The protocol does not store the comparison or the analysis.

## Step 5: design and identification

`LongPanel` reads the long panel. It names the unit, the time, the treatment, and the baseline
covariate. The argument `time_varying` lists the columns that change by week. The argument
`censoring` names the consent column. The argument `treatment_probabilities` maps each offer level
to its logged-probability column. That mapping declares the offer mechanism, so the fit runs no
offer model.

A plan is a tuple with one entry for each week, or a `DynamicRegimen`. The rule of a
`DynamicRegimen` reads the history of that week. The declaration `rule_kind="known"` says that the
rule is a fixed function of each customer's own history, chosen before the fit. The `each_node`
helper applies one function at every week. The typed `RegimeContrast` compares each plan with
`none`.

In [5]:
from cleverly import CausalStudy, LongPanel, RegimeContrast
from cleverly.datasets import OFFER_WEEKS, win_back, win_back_then_ship
from cleverly.longitudinal import DynamicRegimen, each_node

LOGGED = {"none": "p_none", "discount": "p_discount", "shipping": "p_shipping"}


def offers_design(*, logged=True, time_varying=("engaged",)):
    return LongPanel(
        unit="customer",
        time="week",
        treatment="offer",
        baseline=["tenure"],
        time_varying=list(time_varying),
        censoring="consent",
        outcome="full_price",
        treatment_probabilities=LOGGED if logged else None,
    )


def rule(label, offer_for):
    return DynamicRegimen(
        label,
        each_node(lambda history, week: offer_for(history[f"engaged@{week}"]), OFFER_WEEKS),
        rule_kind="known",
    )


study = CausalStudy(panel, design=offers_design(), protocol=protocol)
for week in (1, 2, 3):
    print(f"history_names({week}):", study.data.history_names(week))
print()
plans = {
    "none": ("none",) * OFFER_WEEKS,
    "discount": ("discount",) * OFFER_WEEKS,
    "shipping": ("shipping",) * OFFER_WEEKS,
    "win_back": rule("win_back", win_back),
    "win_back_ship": rule("win_back_ship", win_back_then_ship),
}
effect = study.identify(RegimeContrast(plans, reference="none"))
print(effect.summary(protocol="fingerprint"))

history_names(1): ('tenure', 'engaged@1')
history_names(2): ('tenure', 'engaged@1', 'engaged@2')
history_names(3): ('tenure', 'engaged@1', 'engaged@2', 'engaged@3')

contrast of each regime against the reference
identified by explicit-adjustment: sequential g-formula under the declared treatment regimen
history at offer@1: ['tenure', 'engaged@1']
history at offer@2: ['tenure', 'engaged@1', 'engaged@2', 'offer@1']
history at offer@3: ['tenure', 'engaged@1', 'engaged@2', 'engaged@3', 'offer@1', 'offer@2']
required nuisances: ['sequential_outcome_regressions', 'treatment_and_censoring_mechanisms']
assumptions:
  - consistency: each observed history equals the potential history under its realized regimen
  - no interference: one unit's potential history does not depend on other units' regimens
  - sequential exchangeability given the recorded history at every node
  - sequential positivity for treatment and remaining under observation
causal study protocol: schema 1; 94024dfd011817af


**What this output tells you.** The three `history_names` lines list the columns that the design
puts in the history of each week. Week 1 holds `tenure` and `engaged@1`. Each later week adds that
week's engagement. The earlier offers join the history through the plan, and the `history at` lines
of the summary show them.

The summary names the estimand and its identification, the sequential g-formula under the declared
treatment regimen. The required nuisances are the sequential outcome regressions and the
treatment and censoring mechanisms. The `history at offer@3` line lists `engaged@1`, `engaged@2`,
`engaged@3`, `offer@1`, and `offer@2`, with `tenure`. The last line prints the fingerprint from Step 4.

The assumptions keep the longitudinal shape.

| assumption | what it becomes here | can the data check it? |
| --- | --- | --- |
| sequential exchangeability | each week's offer and consent are independent of the potential outcomes, given the recorded history | the offers hold by design, because the CRM drew them at random with the logged probabilities. The consent part does not hold by design |
| sequential positivity | at every week, each reachable history has a positive probability of the plan's offer and of staying consenting | partly, through the support report in Step 10 |
| consistency | each offer is the declared template, and a code expires at the end of its week | no |
| no interference | one customer's offers do not change another customer's outcome | no |

The risk in this design is consent loss. The fit assumes that consent loss depends only on the
recorded history. Step 13 aims its benchmark at that assumption.

## Step 6: estimate the contrasts with LTMLE

The configuration is written out in full. The offer mechanism is declared, so the fit needs no
offer learner.

| setting | value | what it does |
| --- | --- | --- |
| `outcome_learner` | logistic regression | fits the regression of the binary outcome at the last node |
| `pseudo_learner` | linear regression | fits the intermediate regressions. Their targets are continuous, and the recursion clips their predictions to the unit interval |
| `censoring_learner` | logistic regression | fits the probability of staying consenting at each node, given the history |
| `CrossFitting(n_folds=2)` | two outer folds | evaluates each nuisance on rows it did not fit |
| `Runtime(random_state=42, n_jobs=1)` | fixed seed, one process | makes the fit reproducible |

The code then compares each contrast with its true value. The helper `against_truth` returns the
estimate, the standard error, the interval, and whether the interval contains the truth.

In [6]:
from cleverly import CrossFitting, ModelSpec, Runtime, TMLEMethod

logistic = LogisticRegression(max_iter=1000, random_state=42)
ltmle = TMLEMethod(
    models=ModelSpec(
        outcome_learner=logistic,
        pseudo_learner=LinearRegression(),
        censoring_learner=logistic,
    ),
    cross_fitting=CrossFitting(n_folds=2),
    runtime=Runtime(random_state=42, n_jobs=1),
)
result = effect.estimate(method=ltmle)
print(result.summary(protocol="fingerprint"))


def against_truth(fit):
    rows = []
    for key, estimate in fit.estimates.items():
        low, high = estimate.ci
        rows.append(
            {
                "contrast": key.removeprefix("ate_regimen[").removesuffix(" vs none]"),
                "estimate": estimate.psi,
                "std_error": estimate.std_error,
                "ci_low": low,
                "ci_high": high,
                "truth": truth[key],
                "covers": bool(low <= truth[key] <= high),
            }
        )
    return pd.DataFrame(rows).set_index("contrast")


print()
print(against_truth(result).round(3).to_string())

Longitudinal TMLE (3 time points, n = 20000)

parameter                           estimate  std. error  95% CI             p-value
----------------------------------  --------  ----------  -----------------  -------
ate_regimen[discount vs none]       -0.0464   0.0287      [-0.1028, 0.0099]  0.1061 
ate_regimen[shipping vs none]       0.0437    0.0276      [-0.0104, 0.0978]  0.1132 
ate_regimen[win_back vs none]       0.0701    0.0195      [0.0318, 0.1083]   0.0003 
ate_regimen[win_back_ship vs none]  0.0844    0.0229      [0.0396, 0.1292]   0.0002 

  time points: 3
  long panel: unit column 'customer', decision times 1, 2, 3 (column 'week')
  outcome family: binomial
  regimens: none=(none/none/none), discount=(discount/discount/discount), shipping=(shipping/shipping/shipping), win_back=(d:d@1/d:d@2/d:d@3), win_back_ship=(d:d@1/d:d@2/d:d@3)
    a 'd' is a rule d_t(H_t) read off [W, L_1, ..., L_t]; its followers are a covariate-dependent set, so the counts below describe this sample
 

**What this output tells you.** The table reports four contrasts against `none`. The win-back
contrast is 0.0701, with a standard error of 0.0195 and a 95% interval of (0.0318, 0.1083). The
interval contains the true contrast of 0.0722.

The discount contrast is -0.0464 with a standard error of 0.0287. The shipping contrast is 0.0437
with 0.0276. The win-back-then-ship contrast is 0.0844 with 0.0229. All four intervals contain
their true values on this draw. That is one draw, not a coverage result.

| summary line | what it says on this fit |
| --- | --- |
| `cross-fitting` | `2 fold(s)` |
| `g_bounds` | each cumulative treatment-and-censoring probability is bounded below by 0.01, the package default |
| `known factors` | `offer@1`, `offer@2`, and `offer@3` come from the declared columns, and no bound moved a declared value |
| `causal study protocol` | the fingerprint `94024dfd011817af` from Step 4 |
| the five plan lines | 718, 427, and 525 customers followed `none`, `discount`, and `shipping` throughout. The two rules have 1270 and 928 followers |

The simultaneous bands use a critical value of 2.451 in place of 1.960. They cover the four
contrasts together. The final table repeats the estimates beside the truth and the `covers` flag.

## Step 7: three more estimators on the same plans

The same effect object accepts any sequential method. Each fit below reuses the plans, the design,
and the protocol. The method `SDRMethod` (sequentially doubly robust) uses the learners and folds
of Step 6, and it skips the fluctuation step. The method `GComputationMethod` fits the outcome
regressions with a main-terms `GLM` and no cross-fitting. The method `IPWMethod` weights by the
declared offer probabilities and a fitted consent model. [Sequential g-computation and IPW](../technical-reference/sequential-baselines.md)
states these two estimators.

The code prints each summary, and the next step compares them.

In [7]:
from cleverly import SDRMethod

sdr = effect.estimate(
    method=SDRMethod(models=ltmle.models, cross_fitting=ltmle.cross_fitting, runtime=ltmle.runtime)
)
print(sdr.summary(protocol="fingerprint"))

Longitudinal SDR (3 time points, n = 20000)

parameter                           estimate  std. error  95% CI             p-value
----------------------------------  --------  ----------  -----------------  -------
ate_regimen[discount vs none]       -0.0468   0.0287      [-0.1031, 0.0096]  0.1037 
ate_regimen[shipping vs none]       0.0437    0.0276      [-0.0104, 0.0978]  0.1132 
ate_regimen[win_back vs none]       0.0696    0.0195      [0.0314, 0.1079]   0.0004 
ate_regimen[win_back_ship vs none]  0.0841    0.0229      [0.0393, 0.1289]   0.0002 

  estimator: sequentially doubly robust (Diaz, Williams, Hoffman and Schenck 2023, Section 5.3), no fluctuation
  long panel: unit column 'customer', decision times 1, 2, 3 (column 'week')
  time points: 3
  outcome family: binomial
  regimens: none=(none/none/none), discount=(discount/discount/discount), shipping=(shipping/shipping/shipping), win_back=(d:d@1/d:d@2/d:d@3), win_back_ship=(d:d@1/d:d@2/d:d@3)
    a 'd' is a rule d_t(H_t) read 

**What this output tells you.** The SDR estimates equal the LTMLE estimates within 0.0005. The win-back
contrast is 0.0696 with a standard error of 0.0195. SDR and LTMLE use the same outcome regressions,
the same declared offer factors, and the same folds. They differ by the targeting step. The header
names the method: `sequentially doubly robust`, with `no fluctuation`.

In [8]:
from cleverly import GComputationMethod
from cleverly.learners import GLM

gcomp = effect.estimate(
    method=GComputationMethod(
        models=ModelSpec(outcome_learner=GLM(), pseudo_learner=GLM(), censoring_learner=GLM()),
        runtime=ltmle.runtime,
    )
)
print(gcomp.summary(protocol="fingerprint"))

Longitudinal g-computation (3 time points, n = 20000)

parameter                           estimate  std. error  95% CI             p-value
----------------------------------  --------  ----------  -----------------  -------
ate_regimen[discount vs none]       -0.0479   0.0309      [-0.1085, 0.0126]  0.1206 
ate_regimen[shipping vs none]       0.0447    0.0307      [-0.0154, 0.1048]  0.1450 
ate_regimen[win_back vs none]       0.0699    0.0219      [0.0269, 0.1128]   0.0014 
ate_regimen[win_back_ship vs none]  0.0851    0.0248      [0.0364, 0.1338]   0.0006 

  estimator: sequential g-computation, the iterated conditional expectation plug-in (Bang and Robins 2005, Section 4), no fluctuation; an interval assumes every outcome GLM is correctly specified
  long panel: unit column 'customer', decision times 1, 2, 3 (column 'week')
  time points: 3
  outcome family: binomial
  regimens: none=(none/none/none), discount=(discount/discount/discount), shipping=(shipping/shipping/shipping), win_

**What this output tells you.** The g-computation fit gives 0.0699 for the win-back contrast, with a
standard error of 0.0219. Its header says `no fluctuation` and `an interval assumes every outcome
GLM is correctly specified`. The summary line `cross-fitting` reads `nuisances fitted in sample`.
The declared offer factors enter the support lines only. The point estimate reads the
outcome regressions.

In [9]:
from cleverly import IPWMethod

ipw = effect.estimate(
    method=IPWMethod(models=ModelSpec(censoring_learner=GLM()), runtime=ltmle.runtime)
)
print(ipw.summary(protocol="fingerprint"))

Longitudinal IPW (3 time points, n = 20000)

parameter                           estimate  std. error  95% CI              p-value
----------------------------------  --------  ----------  ------------------  -------
ate_regimen[discount vs none]       -0.0660   0.0328      [-0.1302, -0.0018]  0.0439 
ate_regimen[shipping vs none]       0.0524    0.0302      [-0.0069, 0.1117]   0.0832 
ate_regimen[win_back vs none]       0.0613    0.0219      [0.0184, 0.1041]    0.0051 
ate_regimen[win_back_ship vs none]  0.0698    0.0256      [0.0196, 0.1199]    0.0064 

  estimator: inverse probability weighting, Hajek (normalized by the weight sum); an interval assumes every treatment and censoring GLM is correctly specified
  long panel: unit column 'customer', decision times 1, 2, 3 (column 'week')
  time points: 3
  outcome family: binomial
  regimens: none=(none/none/none), discount=(discount/discount/discount), shipping=(shipping/shipping/shipping), win_back=(d:d@1/d:d@2/d:d@3), win_back_ship=(

**What this output tells you.** The IPW fit gives 0.0613 for the win-back contrast, with a standard
error of 0.0219. The header says `Hajek (normalized by the weight sum)`, and `an interval assumes
every treatment and censoring GLM is correctly specified`. The estimate reads the declared offer
probabilities and the fitted consent model. It reads no outcome regression.

## Step 8: four estimators side by side

The code stacks the four sets of estimates and standard errors. It prints the largest distance of
each fit from the LTMLE estimate.

In [10]:
fits = {"LTMLE": result, "SDR": sdr, "g-computation": gcomp, "IPW": ipw}
side = pd.DataFrame(
    {name: [fit[key].psi for key in result.estimates] for name, fit in fits.items()},
    index=[key.removeprefix("ate_regimen[").removesuffix(" vs none]") for key in result.estimates],
)
side["truth"] = [truth[key] for key in result.estimates]
print("estimates:")
print(side.round(3).to_string())
errors = pd.DataFrame(
    {name: [fit[key].std_error for key in result.estimates] for name, fit in fits.items()},
    index=side.index,
)
print()
print("standard errors:")
print(errors.round(3).to_string())
print()
print("largest distance from the LTMLE estimate:")
for name, fit in fits.items():
    gap = max(abs(fit[key].psi - result[key].psi) for key in result.estimates)
    print(f"  {name:14s} {gap:.4f}   interval status: {fit.inference_status}")

estimates:
               LTMLE    SDR  g-computation    IPW  truth
discount      -0.046 -0.047         -0.048 -0.066 -0.068
shipping       0.044  0.044          0.045  0.052  0.059
win_back       0.070  0.070          0.070  0.061  0.072
win_back_ship  0.084  0.084          0.085  0.070  0.099

standard errors:
               LTMLE    SDR  g-computation    IPW
discount       0.029  0.029          0.031  0.033
shipping       0.028  0.028          0.031  0.030
win_back       0.020  0.020          0.022  0.022
win_back_ship  0.023  0.023          0.025  0.026

largest distance from the LTMLE estimate:
  LTMLE          0.0000   interval status: influence_curve
  SDR            0.0004   interval status: influence_curve
  g-computation  0.0015   interval status: influence_curve
  IPW            0.0196   interval status: influence_curve


**What this output tells you.** On this draw the first three estimators agree. SDR lies within 0.0004
of LTMLE on every contrast, and g-computation within 0.0015. IPW lies 0.0196 away at its
largest. Its win-back estimate is 0.061, against 0.070 for LTMLE and a true 0.072. IPW has a
wider standard error than LTMLE on all four contrasts. The interval status of every fit is `influence_curve`.

The agreement is a fact about this draw and this law. The table states what each estimator reads,
and when its answer holds. It does not rank the estimators.

| estimator | what it reads | consistent when | what its interval assumes |
| --- | --- | --- | --- |
| LTMLE | outcome regressions, declared offers, fitted consent | the outcome regressions or the consent model are correct, because the offer factors are exact | the influence-curve interval of a cross-fitted, targeted fit |
| SDR | the same, without the fluctuation | the same condition | the influence-curve interval of a cross-fitted fit with no fluctuation |
| g-computation | outcome regressions only | each outcome regression is correct | every outcome `GLM` is correct, as the header says |
| IPW | declared offers and fitted consent only | the consent model is correct | every treatment and censoring `GLM` is correct, as the header says |

In this design, the logged probabilities depend only on that week's engagement, which every model
here includes. The outcome regressions are main-terms models, and the law adds an
engagement-tenure interaction. The regressions approximate the law, and on this draw they land
close to LTMLE.

The estimates can differ where the models fail. The registered negative control
`misspecification_control` on law `X` of
[sequential g-computation and IPW](../technical-reference/method-evidence/sequential-baselines.md)
makes every main-terms `GLM` wrong. The fits then read a standardized bias of 1.7 to 2.0 and a
coverage of 0.48 to 0.61. The standard error ratio stays near 1.00, so a correct standard error
does not protect an interval around a wrong estimate.

A review probe, `reviews/notebook-review/probes/sequential-offers/sweep.py`, repeats the fits of
this page on 100 draws. It is not a registered study. The largest distance from the LTMLE estimate
had a median of 0.0005 for SDR, 0.0036 for g-computation, and 0.0126 for IPW. The maxima were
0.0015, 0.0101, and 0.0364. IPW sits above its median on this draw.

## Step 9: the failure mode, a comparison that ignores the shared sample

The question asks which policy beats which. Two contrasts against `none` come from the same
customers, so their estimates move together. The difference of two contrasts needs the joint
influence curve. The call `result.contrast(...)` computes it. The code compares three pairs and prints the
paired standard error beside the standard error of two independent estimates, `sqrt(se_a^2 +
se_b^2)`. The second number is the failure mode.

In [11]:
pairs = [
    ("win_back", "discount"),
    ("win_back", "shipping"),
    ("win_back_ship", "win_back"),
]
rows = []
for better, other in pairs:
    keys = [f"ate_regimen[{better} vs none]", f"ate_regimen[{other} vs none]"]
    paired = result.contrast(
        lambda values: values[0] - values[1], keys, name=f"{better} minus {other}"
    )
    unpaired_se = float(np.hypot(result[keys[0]].std_error, result[keys[1]].std_error))
    low, high = paired.ci
    rows.append(
        {
            "comparison": f"{better} - {other}",
            "estimate": paired.psi,
            "paired_se": paired.std_error,
            "paired_low": low,
            "paired_high": high,
            "unpaired_se": unpaired_se,
            "truth": truth[keys[0]] - truth[keys[1]],
        }
    )
comparison = pd.DataFrame(rows).set_index("comparison")
print(comparison.round(3).to_string())

                          estimate  paired_se  paired_low  paired_high  unpaired_se  truth
comparison                                                                                
win_back - discount          0.117      0.027       0.064        0.169        0.035  0.140
win_back - shipping          0.026      0.025      -0.023        0.076        0.034  0.013
win_back_ship - win_back     0.014      0.020      -0.025        0.054        0.030  0.027


**What this output tells you.** Win-back against a discount every week has an estimate of 0.117.
Its paired standard error is 0.027, and its 95% interval is (0.064, 0.169). The true difference is
0.140. The unpaired standard error is 0.035. The paired interval excludes zero.

Win-back against free shipping has an estimate of 0.026, a paired standard error of 0.025, and an
interval of (-0.023, 0.076). Win-back then ship against win-back has an estimate of 0.014, a
standard error of 0.020, and an interval of (-0.025, 0.054). The unpaired standard errors are 0.034
and 0.030. Both paired intervals contain zero. The true differences are 0.013 and 0.027.
This draw cannot separate the three better policies.

The unpaired number is larger in every row. It treats two estimates that share every customer as
independent, and it ignores their positive correlation. A program that used it would wait for more
customers than it needs. In the review probe, the unpaired standard error was larger than the
paired one on 100 of 100 draws. The paired interval for win-back against a discount every week
excluded zero on 100 of 100 draws. For the other two pairs it excluded zero on 7 and 25 draws of
100. The sample is sized for the first question and not for the other two.

## Step 10: diagnostics, what the fit can check

`result.assess()` runs the checks that apply to the fit. The code prints the operations that ran.
It then prints the support report. The report shows how many customers followed each plan at
each node, how heavy their weights are, and how many weights a bound replaced. A weight is one over the cumulative
probability of following the plan and staying consenting. The code ends with the consent model of
each node, measured on rows it did not fit.

In [12]:
assessment = result.assess()
ledger = assessment.to_frame()
ran = ledger[~ledger["status"].isin(["unavailable", "not_applicable", "deferred"])]
print(ran[["surface", "check", "status"]].to_string(index=False))
support = assessment.report("support").to_frame()
columns = ["regimen", "time", "n_followed", "max_weight", "effective_n", "share_truncated"]
print()
print(support[columns].round(3).to_string(index=False))
kish_ratio = (support["effective_n"] / support["n_followed"]).min()
print(f"minimum effective-sample-size ratio: {kish_ratio:.1%}")
print()
nuisance = assessment.report("nuisance_models").to_frame()
consent = nuisance[nuisance["role"] == "censoring"]
print(
    consent[["role", "time", "n", "evaluation", "auc", "calibration_slope"]]
    .round(3)
    .to_string(index=False)
)

   surface           check    status
validation score_equations    passed
validation         support completed
validation nuisance_models completed

      regimen  time  n_followed  max_weight  effective_n  share_truncated
         none     1        6836       4.000     6655.279              0.0
         none     2        2233      16.148     2080.153              0.0
         none     3         718      60.032      633.853              0.0
     discount     1        6456       4.351     5999.630              0.0
     discount     2        1698      18.569     1518.202              0.0
     discount     3         427      75.712      367.797              0.0
     shipping     1        5688       4.805     5494.639              0.0
     shipping     2        1749      22.134     1602.542              0.0
     shipping     3         525      92.840      462.409              0.0
     win_back     1        8122       2.714     8109.031              0.0
     win_back     2        3229      

**What this output tells you.** The first table lists the operations that ran. The score-equation
check is `passed`. The support report and the nuisance report are `completed`. A completed result
means the calculation ran. It is not a pass.

The support table divides by the declared offer factors and the fitted consent model. It describes
those models. It cannot see an unrecorded cause of consent loss.

| plan | followers at week 3 | max weight | effective n | what it shows |
| --- | --- | --- | --- | --- |
| `none` | 718 | 60.032 | 633.853 | thin support |
| `discount` | 427 | 75.712 | 367.797 | thinnest support of the static plans |
| `shipping` | 525 | 92.840 | 462.409 | the heaviest weight |
| `win_back` | 1270 | 19.399 | 1264.961 | the rule keeps the most customers |
| `win_back_ship` | 928 | 27.744 | 907.677 | the rule keeps more than any static plan |

A static plan sends the same offer to engaged and unengaged customers. The logged probability of
that offer is lower in one of the two groups, so few customers follow the plan. Win-back sends the
most likely logged offer in each group, so it keeps the most customers.

The minimum ratio of effective sample size to followers is 86.1%. No row has a truncated weight at
the default bound of `(0.01, 1)`, which caps a weight at 100. In the review probe, that held on 54
of 100 draws. On the others, the largest weight reached the cap of 100.

The consent models have an `auc` of 0.633, 0.649, and 0.651 at weeks 1, 2, and 3. They separate the
customers poorly, because the chance of consent loss varies little with engagement and
tenure. The out-of-fold calibration slopes are 0.971, 0.977, and 0.923. A slope below 1 means
the predictions spread wider than the outcomes support. These numbers describe the consent model
and do not show that it is correct.

## Step 11: a truncation curve, with the logs unavailable

The declared fit shows no truncation curve. The curve replays bounds that the fit would refuse on
a declared design mechanism. This step runs a secondary analysis. It treats the logged
probabilities as unavailable and estimates the offer mechanism from the history with a seeded
logistic regression. It then calls `truncation_curve` on that fit.

This analysis is a different estimator, not a sensitivity interval, and the curve does not pick a
preferred bound. It is descriptive. The code prints three tables. They give the estimate at each bound, the move in
standard errors of the fit, and the number of score cells that each bound truncated.

In [13]:
logs_missing = CausalStudy(panel, design=offers_design(logged=False), protocol=protocol)
estimated = TMLEMethod(
    models=ModelSpec(
        outcome_learner=logistic,
        pseudo_learner=LinearRegression(),
        treatment_learner=LogisticRegression(max_iter=1000, random_state=42),
        censoring_learner=logistic,
    ),
    cross_fitting=ltmle.cross_fitting,
    runtime=ltmle.runtime,
)
no_logs = logs_missing.identify(RegimeContrast(plans, reference="none")).estimate(method=estimated)
print("offer mechanism: estimated from the history; no logged probabilities are declared")
print(against_truth(no_logs).round(3).to_string())
print()
curve = no_logs.diagnostics.truncation_curve(bounds=[0.01, 0.02, 0.05, 0.10])
curve["contrast"] = curve["estimand"].str.removeprefix("ate_regimen[").str.removesuffix(" vs none]")
by_bound = {
    "estimate": "psi",
    "move": "delta_from_fitted",
    "cells": "truncated_score_cells",
    "evaluated": "evaluated_score_cells",
}
tables = {
    name: curve.pivot(index="contrast", columns="lower_bound", values=column)
    for name, column in by_bound.items()
}
no_logs_se = pd.Series(
    {
        key.removeprefix("ate_regimen[").removesuffix(" vs none]"): no_logs[key].std_error
        for key in no_logs.estimates
    }
)
print("estimate at each lower bound:")
print(tables["estimate"].round(4).to_string())
print()
print("move from the fitted bound, in standard errors:")
print(tables["move"].div(no_logs_se, axis=0).round(3).to_string())
print()
print("score cells truncated, and score cells evaluated:")
cells = (
    tables["cells"].astype(int).astype(str) + " of " + tables["evaluated"].astype(int).astype(str)
)
print(cells.to_string())

offer mechanism: estimated from the history; no logged probabilities are declared
               estimate  std_error  ci_low  ci_high  truth  covers
contrast                                                          
discount         -0.046      0.030  -0.106    0.013 -0.068    True
shipping          0.042      0.029  -0.016    0.099  0.059    True
win_back          0.070      0.020   0.030    0.110  0.072    True
win_back_ship     0.084      0.024   0.037    0.130  0.099    True



estimate at each lower bound:
lower_bound      0.01    0.02    0.05    0.10
contrast                                     
discount      -0.0464 -0.0433 -0.0442 -0.0443
shipping       0.0417  0.0397  0.0413  0.0416
win_back       0.0699  0.0703  0.0693  0.0700
win_back_ship  0.0838  0.0841  0.0848  0.0845

move from the fitted bound, in standard errors:
lower_bound    0.01   0.02   0.05   0.10
contrast                                
discount        0.0  0.101  0.073  0.071
shipping        0.0 -0.067 -0.010 -0.003
win_back        0.0  0.017 -0.031  0.003
win_back_ship   0.0  0.014  0.043  0.028

score cells truncated, and score cells evaluated:
lower_bound          0.01          0.02           0.05           0.10
contrast                                                             
discount       0 of 18368  160 of 18368   859 of 18368  2275 of 18368
shipping       0 of 17749  125 of 17749   997 of 17749  2507 of 17749
win_back       0 of 22408   36 of 22408   468 of 22408  2453 of 2240

**What this output tells you.** The offer mechanism is estimated, so the estimates differ from
Step 6 by the estimation of the offer model. The win-back contrast is 0.070 with a standard error
of 0.020. All four intervals contain their true values on this draw.

The first table gives the estimate at the lower bounds 0.01, 0.02, 0.05, and 0.10. The bound 0.01
is the fit itself. The second table divides each move from the fitted bound by the standard error
of the fit. The largest move is 0.101 standard errors, for the discount contrast at the bound 0.02.

The third table counts the score cells that a bound truncated. A score cell is one followed customer
at one node of one plan. At the bound 0.10, the discount contrast truncates 2275 of 18368 cells,
and the estimate moves 0.071 standard errors. A truncated cell has a cumulative
probability below the bound, so it carries a weight above 10. In the review probe, the largest
move over 100 draws was 0.513 standard errors. That maximum covers the bounds 0.02, 0.05, and
0.10 and the four contrasts.

The curve shows stability to this joint truncation choice. It does not isolate the offer model,
and it does not validate the estimated mechanism.

## Step 12: a synthetic deployment check

A real campaign would run the win-back policy on a pilot cohort and compare its observed rate with
the estimate. No applied data access exists for this tutorial, so the pilot is synthetic. The
`policy="win_back"` argument draws a fresh cohort of 10000 customers under that policy. In the
cohort, every customer consents and stays tracked. This is a design condition of the pilot: a
consented panel with no loss.

The code fits the win-back level with `RegimeMean` and compares it with the pilot mean. The pilot
is independent of the estimation sample, so the standard error of the difference is
`sqrt(se_fit^2 + se_pilot^2)`.

In [14]:
from cleverly import RegimeMean

pilot, _ = make_sequential_offers(10_000, seed=100_003, policy="win_back")
pilot_outcome = pilot.loc[pilot["week"] == 3, "full_price"]
levels = study.identify(RegimeMean(plans)).estimate(method=ltmle)
fitted = levels["ey_regimen[win_back]"]
pilot_mean = float(pilot_outcome.mean())
pilot_se = float(pilot_outcome.std() / np.sqrt(len(pilot_outcome)))
difference = fitted.psi - pilot_mean
difference_se = float(np.hypot(fitted.std_error, pilot_se))
print(
    f"pilot customers: {len(pilot_outcome)}; lost to consent: {int((pilot['consent'] == 0).sum())}"
)
print(f"pilot share of full-price orders:  {pilot_mean:.3f}  (standard error {pilot_se:.4f})")
print(
    f"fitted win-back level:             {fitted.psi:.3f}  (standard error {fitted.std_error:.4f})"
)
print(f"difference, fit minus pilot:       {difference:+.4f}  (standard error {difference_se:.4f})")
print(f"difference in standard errors:     {difference / difference_se:+.2f}")
print(f"population win-back level:         {truth['ey_regimen[win_back]']:.3f}")

pilot customers: 10000; lost to consent: 0
pilot share of full-price orders:  0.603  (standard error 0.0049)
fitted win-back level:             0.608  (standard error 0.0133)
difference, fit minus pilot:       +0.0050  (standard error 0.0142)
difference in standard errors:     +0.35
population win-back level:         0.600


**What this output tells you.** The pilot has 10000 customers and loses none to consent. Its full-price
rate is 0.603, with a standard error of 0.0049. The fitted win-back level is 0.608, with a standard
error of 0.0133. The difference is +0.0050 with a standard error of 0.0142, which is +0.35
standard errors. The population value is 0.600.

The two numbers agree within sampling error on this draw. The check is informative because the pilot
does not use the logs. If the exchangeability or the consent assumption failed, the fit could
move away from the pilot. A small difference does not prove either assumption. In the
review probe, the absolute difference stayed below 1.96 standard errors on 93 of 100 draws.

## Step 13: how much a recorded cause moves the estimate

The offers hold exchangeability by design, because the CRM drew them at random with the logged
probabilities. The assumption at risk is consent loss. The fit assumes that consent loss depends
only on the recorded history. Engagement is the recorded cause of consent loss in this law, with
`tenure`.

No omitted-variable bound is implemented for a longitudinal fit, so this step uses a benchmark.
The code refits the two static contrasts without `engaged` in the history. The declared offer
probabilities stay the same in both fits. The move then comes from the outcome and consent models.
The win-back rules read `engaged`, so they cannot run without it. The covariate `tenure` is the only baseline
covariate, so the code does not drop it.

In [15]:
static = {name: plans[name] for name in ("none", "discount", "shipping")}
full_fit = study.identify(RegimeContrast(static, reference="none")).estimate(method=ltmle)
reduced_study = CausalStudy(panel, design=offers_design(time_varying=()), protocol=protocol)
reduced_fit = reduced_study.identify(RegimeContrast(static, reference="none")).estimate(
    method=ltmle
)
rows = []
for key, estimate in full_fit.estimates.items():
    move = reduced_fit[key].psi - estimate.psi
    rows.append(
        {
            "contrast": key.removeprefix("ate_regimen[").removesuffix(" vs none]"),
            "with_engagement": estimate.psi,
            "without": reduced_fit[key].psi,
            "move": move,
            "move_in_se": move / estimate.std_error,
        }
    )
benchmark = pd.DataFrame(rows).set_index("contrast")
print(benchmark.round(3).to_string())

          with_engagement  without   move  move_in_se
contrast                                             
discount           -0.046   -0.048 -0.002      -0.058
shipping            0.044    0.041 -0.002      -0.083


**What this output tells you.** The full history gives -0.046 for the discount contrast and 0.044
for the shipping contrast. Dropping `engaged` moves them to -0.048 and 0.041. The moves are -0.002
and -0.002, which are -0.058 and -0.083 standard errors.

Read the table as a benchmark, not as a bound. A move describes the dropped covariate only. An
unrecorded cause of consent loss can act through other paths, and its move can be larger. The move
is small here, and the conclusion about static plans stays unchanged. In the review probe, the
move stayed below one standard error on 100 of 100 draws for both contrasts.

## How far to trust this

No registered study covers this page's law, which has three offer levels, three nodes, a declared
offer mechanism, estimated consent loss, and `n_folds=2`. Three sources cover the nearest
constructions. None covers all of those features at once.

| source | what it covers |
| --- | --- |
| [LTMLE on declared known node mechanisms](../technical-reference/method-evidence/known-node-mechanisms-ltmle.md), study `known-node-mechanisms-ltmle` | a two-node SMART with a binary treatment, in sample, with declared factors and with estimated censoring |
| `tests/unit/test_sdr_known_mechanisms.py` | SDR on the same SMART law, with 1 or 3 folds, on exact and declared factors |
| `tests/unit/test_sequential_baselines_declared.py` | IPW and g-computation on an exact finite law with a declared three-level offer, an estimated censoring `GLM`, and a mutation control |

| cited cell | law | nuisance construction | outcome type | fold layer | role | result |
| --- | --- | --- | --- | --- | --- | --- |
| accuracy row `ate_regimen[always vs never]`, correct outcome regressions | a two-node SMART with binary offers, n = 2,000, 1,000 draws | treatment and censoring factors declared | binary | in sample | positive | coverage 0.9480, SE ratio 0.9804 |
| `known_mechanism_accuracy/ate_regimen__ltmle_known_treatment__censoring_estimated` | the same law, outcome regressions without `L1` | treatment declared, censoring fitted by a correct logistic regression | binary | in sample | positive | coverage 0.9272 to 0.9644, SE ratio 0.9974 |
| accuracy row `ate_regimen[static_12 vs static_00]` on law `D2` in `sequential-baselines` | two nodes with a three-arm second node, and censoring | every mechanism and regression fitted by a correct `GLM` | binary | in sample | positive | coverage 0.9500 for g-computation and 0.9590 for Hajek IPW, SE ratio 0.9957 and 1.0054 |
| `misspecification_control/hajek_x_contrast__main_terms` | law `X`: three nodes, continuous outcome | every `GLM` wrong | continuous | in sample | control, red as declared | bias 0.1306 to 0.1344, coverage 0.5072 to 0.5348 |

The values come from the `summary.csv`, `properties.csv`, and ledger rows of the cited pages.
The cited cells use binary offers, or a three-arm node at one decision. They run in sample, or
with 1 or 3 folds, and never with 2. This page runs three offer levels at three nodes with 2 folds, which none of them covers.
The exact-law tests give the three-level declared composition for IPW and g-computation at two
nodes.

A review probe, `reviews/notebook-review/probes/sequential-offers/sweep.py`, repeats the fits of
this page on 100 draws of this law, seeds 1000 to 1099. It is not a registered study. The LTMLE
interval covered the true contrast on 94, 95, 93, and 95 of 100 draws for the discount,
shipping, win-back, and win-back-then-ship contrasts.

The mean standard error over the empirical spread ran from 0.973 to 1.048. The win-back estimate had a bias of -0.0063 with a Monte Carlo
standard error of 0.0020. The other three estimators showed the same sign and size. The probe did
not isolate the source of this bias.

| layer | establishes | does not establish |
| --- | --- | --- |
| the support report | how many customers followed each plan, and how heavy the weights are | that sequential exchangeability holds for consent loss |
| the score-equation report | each node solved its equation | that the node regressions are correctly specified |
| the nuisance report | out-of-fold loss and calibration for the consent models | that the consent model is correct |
| the truncation curve | how far a joint bound moves a fit with an estimated offer mechanism | a sensitivity interval, or a preferred bound |
| the covariate benchmark | how far the contrasts move when engagement leaves the history | a bound, or the strength of an unrecorded cause |
| the synthetic deployment check | agreement with a pilot under the same plan, on a synthetic cohort | that the assumptions hold on applied data. No applied data access exists |
| the review probe | the coverage of the Step 6 intervals over 100 draws of this law | a registered result, or the behavior on another law |

Nothing in this list validates the causal reading. That rests on sequential exchangeability for the
consent loss, consistency, and no interference. All three are arguments about the campaign rather
than about the fit.

## Where to go next

This page compared five offer policies against no offers, and three pairs of policies. The
[longitudinal TMLE tutorial](longitudinal-tmle.ipynb) fits two decisions in the health program,
with a binary treatment and learned mechanisms. The
[longitudinal user guide](../user-guide/longitudinal.md) shows how to read a long panel and how to
report the sequential baselines. The [sequential g-computation and
IPW](../technical-reference/sequential-baselines.md) reference states the two baselines.

The [examples index](index.md#the-retailer-program) lists every tutorial in both programs.